In [ ]:
# Core
import pandas as pd
import numpy as np
import time
# Models
from sklearn.tree import DecisionTreeRegressor

# Metrics
from sklearn.metrics import mean_squared_error

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv("data/electricity_market_dataset.csv")
# Convert timestamp
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Sort by time (VERY IMPORTANT)
df = df.sort_values('Timestamp').reset_index(drop=True)

df.head()
print("Shape:", df.shape)


Format the data

In [ ]:
target = "Electricity_Price_Forecast"
df_model = df.copy()

# Encode categorical columns
df_model["Regulatory_Policies"] = df_model["Regulatory_Policies"].map({"Low": 0, "Medium": 1, "High": 2})
df_model["Energy_Access_Data"] = df_model["Energy_Access_Data"].map({"Rural": 0, "Urban": 1})
df_model["Project_Risk_Analysis"] = df_model["Project_Risk_Analysis"].map({"Low": 0, "Medium": 1, "High": 2})


# Drop target
X = df_model.drop(columns=[target, "Timestamp"])
y = df_model[target]

In [ ]:
# Plot Function
def makeRSMEPlot(df_results, vmin=.8, vmax=25, title="RMSE Heatmap"):
    plt.figure(figsize=(8, 6))
    sns.heatmap(df_results.astype(float), annot=True, fmt=".2f", cmap="coolwarm",
                vmin=vmin, vmax=vmax)
    plt.title(title)
    plt.xlabel("Tree Depth")
    plt.ylabel("Look-Back (rows)")
    plt.show()
def makeTimePlot(df_time, vmin=0, vmax=9, title="Computation Time (s)"):
    plt.figure(figsize=(8, 6))
    sns.heatmap(df_time.astype(float), annot=True, fmt=".2f", cmap="coolwarm",
                vmin=vmin, vmax=vmax)
    plt.title(title)
    plt.xlabel("Tree Depth")
    plt.ylabel("Look-Back (rows)")
    plt.show()

Define Static Function

In [ ]:
def train_static(X, y, look_back, depthOfTree):
    X_train = X.iloc[:look_back]
    y_train = y.iloc[:look_back]

    X_test = X.iloc[look_back:]
    y_test = y.iloc[look_back:]

    model = DecisionTreeRegressor(max_depth=depthOfTree)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    return predictions, y_test


Run fist experiment

In [ ]:
# Training window sizes (1, 3, 6, 12, 18, 24 months)
look_back_values = [720, 2160, 4320, 8640,  12960, 17280]  
tree_depths = [3, 5, 7]  

# Dictionary to store RMSE
results_static = {}
time_results_static = {}
for lb in look_back_values:
    results_static[lb] = {} 
    time_results_static[lb] = {}
    for depth in tree_depths:
        start_time = time.time()
        predictions, y_test = train_static(X, y, look_back=lb, depthOfTree=depth)
        elapsed_time = time.time() - start_time
        time_results_static[lb][depth] =elapsed_time
        # Compute RMSE
        error = np.sqrt(mean_squared_error(y_test, predictions))
        
        # Store
        results_static[lb][depth] = error
        
        print(f"Look-back {lb}, Depth {depth} → RMSE: {error:.2f}, Time: {elapsed_time:.2f}s")

In [ ]:
df_static_rmse = pd.DataFrame(results_static).T 
df_static_time = pd.DataFrame(time_results_static).T
makeRSMEPlot(df_results=df_static_rmse, vmax= 28, vmin=20)
makeTimePlot(df_static_time, vmax=.5)

Defining Progressive Training Function

In [ ]:
def train_progressive(X, y, look_back, look_ahead, retrain_step, depthOfTree):
    """
    Progressive (rolling) model:
    - Trains on the first look_back rows
    - Retrains every retrain_step rows
    - Predicts look_ahead rows at each step
    """
    errors = []
    predictions_list = []

    n_rows = X.shape[0]
    
    # Initialize model and first training window
    model = DecisionTreeRegressor(max_depth=depthOfTree)
    X_train = X.iloc[:look_back]
    y_train = y.iloc[:look_back]
    model.fit(X_train, y_train)
    
    # Start predicting from the end of initial training
    start_idx = look_back
    
    while start_idx + look_ahead <= n_rows:
        # Slice test window
        X_test = X.iloc[start_idx:start_idx + look_ahead]
        y_test = y.iloc[start_idx:start_idx + look_ahead]
        
        # Predict
        preds = model.predict(X_test)
        predictions_list.extend(preds)
        
        # Compute error
        error = np.sqrt(mean_squared_error(y_test, preds))
        errors.append(error)
        
        # Move to next step
        start_idx += look_ahead
        
        # Retrain if we passed a retrain_step
        if start_idx % retrain_step == 0:
            # Use most recent look_back rows for retraining
            train_start = max(0, start_idx - look_back)
            X_train = X.iloc[train_start:start_idx]
            y_train = y.iloc[train_start:start_idx]
            model.fit(X_train, y_train)
    
    # Average RMSE across all predictions
    avg_error = np.mean(errors)
    return predictions_list, avg_error

We will use the function to look ahead by 1 month, retraining every 3 months. Using the same training lengths and decision tree sizes

In [ ]:
retrain_step = 24*90 
look_ahead = 24 *30
results_progressive = {}
time_results_progressive = {}

for lb in look_back_values:
    results_progressive[lb] = {}
    time_results_progressive[lb] = {}
    for depth in tree_depths:
        start_time = time.time()
        _, avg_rmse = train_progressive(X, y, look_back=lb, look_ahead=look_ahead,
                                        retrain_step=24*90, depthOfTree=depth)
        elapsed_time = time.time() - start_time
        time_results_progressive[lb][depth] = elapsed_time
        results_progressive[lb][depth] = avg_rmse
        print(f"Look-back {lb}, Depth {depth} → RMSE: {error:.2f}, Time: {elapsed_time:.2f}s")


In [ ]:
df_progressive_rmse = pd.DataFrame(results_progressive).T 
makeRSMEPlot(df_progressive_rmse, vmin=.8, vmax=1.8)

df_progressive_time = pd.DataFrame(time_results_progressive).T
makeTimePlot(df_progressive_time, vmin=.2, vmax=7)

In [ ]:
def train_mistake_driven(X, y, look_back, look_ahead, depthOfTree, threshold):
    """
    Mistake-Driven Decision Tree:
    - Trains on the first look_back rows
    - Predicts in chunks of look_ahead
    - Retrains ONLY if RMSE of last prediction > threshold
    """
    errors = []
    predictions_list = []

    n_rows = X.shape[0]
    
    # Initialize model with initial training window
    model = DecisionTreeRegressor(max_depth=depthOfTree)
    X_train = X.iloc[:look_back]
    y_train = y.iloc[:look_back]
    model.fit(X_train, y_train)
    
    start_idx = look_back
    
    while start_idx + look_ahead <= n_rows:
        # Slice test window
        X_test = X.iloc[start_idx:start_idx + look_ahead]
        y_test = y.iloc[start_idx:start_idx + look_ahead]
        
        # Predict
        preds = model.predict(X_test)
        predictions_list.extend(preds)
        
        # Compute RMSE for this window
        error = np.sqrt(mean_squared_error(y_test, preds))
        errors.append(error)
        
        # Retrain if error exceeds threshold
        if error > threshold:
            train_start = max(0, start_idx - look_back)
            X_train = X.iloc[train_start:start_idx]
            y_train = y.iloc[train_start:start_idx]
            model.fit(X_train, y_train)
        
        # Move forward
        start_idx += look_ahead
    
    avg_error = np.mean(errors)
    return predictions_list, avg_error

In [ ]:
threshold = 1.5           # retrain if RMSE > 3
results_mistake = {}
time_results_mistake = {}

for lb in look_back_values:
    results_mistake[lb] = {}
    time_results_mistake[lb] = {}
    for depth in tree_depths:
        start_time = time.time()
        _, avg_rmse = train_mistake_driven(X, y, look_back=lb, look_ahead=look_ahead,
                                           depthOfTree=depth, threshold=threshold)
        elapsed_time = time.time() - start_time
        
        results_mistake[lb][depth] = avg_rmse
        time_results_mistake[lb][depth] = elapsed_time
        
        print(f"Look-back {lb}, Depth {depth} → RMSE: {avg_rmse:.2f}, Time: {elapsed_time:.2f}s")

In [ ]:
df_mistake_rmse = pd.DataFrame(results_mistake).T  # RMSE
df_mistake_time = pd.DataFrame(time_results_mistake).T  # Computation time
makeRSMEPlot(df_mistake_rmse)
makeTimePlot(df_mistake_time)

In [ ]:
makeRSMEPlot(df_static_rmse, title="Static RMSE")
makeRSMEPlot(df_progressive_rmse, title="Progressive RMSE")
makeRSMEPlot(df_mistake_rmse, title="Mistake-Driven RMSE")

makeTimePlot(df_static_time, title="Static Time (s)")
makeTimePlot(df_progressive_time, title="Progressive Time (s)")
makeTimePlot(df_mistake_time, title="Mistake-Driven Time (s)")


# Conclusion
For the electrical project the time savings seem to have an inverse relationship with the RSME value that comes back. If the retrain is set to a competitive level with progressive window it is possible to get RSME within striking distance struggles to keep things going